In [ ]:
import sys
print(f"Python version: {sys.version}")


In [ ]:
import warnings
import logging
from pathlib import Path
from datetime import datetime, timedelta

# Data handling
import numpy as np
import pandas as pd

# Data fetching
import yfinance as yf

# Machine learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Configuration
CONFIG = {
    # Stock ticker symbol (e.g., "AAPL", "TSLA", "MSFT", "GOOGL")
    "TICKER": "AAPL",

    # Data date range (fetches 5 years of history for robust training)
    "START_DATE": "2019-01-01",
    "END_DATE": datetime.today().strftime("%Y-%m-%d"),

    # Train/test split ratio (0.8 = 80% for training, 20% for testing)
    "TRAIN_RATIO": 0.80,

    # Feature engineering parameters
    "LAG_DAYS": [1, 2, 3, 5, 10],           # How many days back to create lag features
    "ROLLING_WINDOWS": [5, 10, 20],          # Rolling average window sizes (days)

    # Random Forest hyperparameters
    "RF_N_ESTIMATORS": 200,                  # Number of decision trees in the forest
    "RF_MAX_DEPTH": 10,                      # Maximum depth of each tree (prevents overfitting)
    "RF_MIN_SAMPLES_LEAF": 5,               # Min samples required at leaf node
    "RF_RANDOM_STATE": 42,                   # Seed for reproducibility

    # Output directories (using pathlib for OS-agnostic paths)
    "DATA_DIR": Path("data"),
    "OUTPUT_DIR": Path("outputs"),
}

# Environment setup

# Suppress noisy warnings that don't affect functionality
warnings.filterwarnings("ignore")

# Configure logging  this is how production code tracks what's happening
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Create output directories if they don't exist
CONFIG["DATA_DIR"].mkdir(exist_ok=True)
CONFIG["OUTPUT_DIR"].mkdir(exist_ok=True)

# Plot style

# Set a professional dark theme for all plots
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

PLOT_CONFIG = {
    "figure.figsize": (16, 8),
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 12,
    "lines.linewidth": 1.8,
}
plt.rcParams.update(PLOT_CONFIG)

# REPRODUCIBILITY

# Setting numpy random seed ensures results are the same every run
np.random.seed(CONFIG["RF_RANDOM_STATE"])

# STARTUP LOG
logger.info("=" * 60)
logger.info("Stock Price Prediction System  Initialized")
logger.info(f"Target Stock   : {CONFIG['TICKER']}")
logger.info(f"Date Range     : {CONFIG['START_DATE']}  {CONFIG['END_DATE']}")
logger.info(f"Output Dir     : {CONFIG['OUTPUT_DIR'].resolve()}")
logger.info("=" * 60)

print("Environment configured successfully.")
print(f"    Stock     : {CONFIG['TICKER']}")
print(f"    From      : {CONFIG['START_DATE']}")
print(f"    To        : {CONFIG['END_DATE']}")


In [ ]:
def fetch_stock_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Download historical OHLCV stock data from Yahoo Finance.

    Parameters
    ----------
    ticker : str
        The stock ticker symbol (e.g., 'AAPL' for Apple Inc.)
    start : str
        Start date in 'YYYY-MM-DD' format
    end : str
        End date in 'YYYY-MM-DD' format

    Returns
    -------
    pd.DataFrame
        DataFrame with DatetimeIndex and columns: Open, High, Low, Close, Volume

    Raises
    ------
    ValueError
        If no data is returned or ticker is invalid
    """
    logger.info(f"Fetching data for {ticker} from {start} to {end}...")

    # yf.download() is the core function  it calls Yahoo Finance's API
    # auto_adjust=True: Adjusts historical prices for stock splits and dividends
    # This is critical  without it, split-adjusted prices look
    # like crashes/spikes in the data (false signals)
    raw_df = yf.download(
        tickers=ticker,
        start=start,
        end=end,
        auto_adjust=True,   # Adjust for splits and dividends
        progress=False,     # Suppress download progress bar
    )

    # Check that the download returned usable rows.
    if raw_df.empty:
        raise ValueError(
            f"No data returned for ticker '{ticker}'. "
            f"Check that the ticker symbol is valid and the date range is correct."
        )

    # yfinance sometimes returns a MultiIndex column (ticker, feature) when
    # downloading multiple stocks. We flatten it to just the feature names.
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df.columns = raw_df.columns.droplevel(1)

    # Keep only the 5 standard OHLCV columns in a consistent order
    ohlcv_columns = ["Open", "High", "Low", "Close", "Volume"]

    # Check all required columns exist
    missing = [col for col in ohlcv_columns if col not in raw_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = raw_df[ohlcv_columns].copy()

    # Ensure the index is a proper DatetimeIndex (essential for time series ops)
    df.index = pd.to_datetime(df.index)
    df.index.name = "Date"

    # Sort chronologically (oldest to newest)
    df.sort_index(inplace=True)

    logger.info(f"Successfully fetched {len(df):,} trading days of data.")
    return df


def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Perform data quality checks and handle issues.

    Checks performed:
    - Missing values (NaN)
    - Duplicate dates
    - Negative prices (physically impossible)
    - Zero volume days

    Parameters
    ----------
    df : pd.DataFrame
        Raw OHLCV DataFrame from fetch_stock_data()

    Returns
    -------
    pd.DataFrame
        Cleaned, validated DataFrame
    """
    logger.info("Running data quality validation...")

    original_rows = len(df)

    # CHECK 1: Missing Values
    null_counts = df.isnull().sum()
    if null_counts.any():
        logger.warning(f"Found missing values:\n{null_counts[null_counts > 0]}")
        # Forward-fill missing values (use the last known value)
        # This is the standard approach for financial time series
        df.ffill(inplace=True)
        # If any NaNs remain at the start (no previous value to fill from), drop them
        df.dropna(inplace=True)
        logger.info(f"Missing values handled via forward-fill. Dropped {original_rows - len(df)} rows.")
    else:
        logger.info(" No missing values found.")

    # CHECK 2: Duplicate Dates
    duplicate_count = df.index.duplicated().sum()
    if duplicate_count > 0:
        logger.warning(f"Found {duplicate_count} duplicate dates  keeping last occurrence.")
        df = df[~df.index.duplicated(keep="last")]
    else:
        logger.info(" No duplicate dates found.")

    # CHECK 3: Negative Prices
    price_cols = ["Open", "High", "Low", "Close"]
    negative_mask = (df[price_cols] < 0).any(axis=1)
    if negative_mask.any():
        logger.warning(f"Found {negative_mask.sum()} rows with negative prices  removing.")
        df = df[~negative_mask]
    else:
        logger.info(" No negative prices found.")

    # CHECK 4: Logical Price Consistency
    # High must be >= Low, High must be >= Open and Close
    invalid_ohlc = (
        (df["High"] < df["Low"]) |
        (df["High"] < df["Open"]) |
        (df["High"] < df["Close"]) |
        (df["Low"] > df["Open"]) |
        (df["Low"] > df["Close"])
    )
    if invalid_ohlc.any():
        logger.warning(f"Found {invalid_ohlc.sum()} rows with invalid OHLC relationships  removing.")
        df = df[~invalid_ohlc]
    else:
        logger.info(" All OHLC price relationships are valid.")

    logger.info(f"Validation complete. Final dataset: {len(df):,} rows.")
    return df


# EXECUTE

# Fetch the data
raw_data = fetch_stock_data(
    ticker=CONFIG["TICKER"],
    start=CONFIG["START_DATE"],
    end=CONFIG["END_DATE"],
)

# Validate and clean the data
df_clean = validate_data(raw_data)

# Save a local copy so we don't need to re-download during development
cache_path = CONFIG["DATA_DIR"] / f"{CONFIG['TICKER']}_raw.csv"
df_clean.to_csv(cache_path)
logger.info(f"Raw data cached to: {cache_path}")

# DISPLAY DATA SUMMARY

print("\n" + "="*60)
print(f"  DATA SUMMARY: {CONFIG['TICKER']}")
print("="*60)
print(f"\nShape         : {df_clean.shape[0]:,} rows  {df_clean.shape[1]} columns")
print(f" Date Range    : {df_clean.index[0].date()}  {df_clean.index[-1].date()}")
print(f" Trading Days  : {len(df_clean):,}")
print(f"\n{''*60}")
print("\n First 5 rows (oldest data):")
display(df_clean.head())

print("\nLast 5 rows (most recent data):")
display(df_clean.tail())

print("\nStatistical Summary:")
display(df_clean.describe().round(2))



In [ ]:
# Exploratory data analysis
# Never skip EDA. Understanding your data is the difference
# between a model that works and one that fails silently.

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.suptitle(
    f"{CONFIG['TICKER']}  Exploratory Data Analysis",
    fontsize=20, fontweight="bold", y=1.01
)

## Plot 1: Full Historical Close Price
ax1 = axes[0, 0]
ax1.plot(df_clean.index, df_clean["Close"], color="#2196F3", linewidth=1.2, alpha=0.9)
ax1.fill_between(df_clean.index, df_clean["Close"], alpha=0.08, color="#2196F3")
ax1.set_title("Historical Closing Price")
ax1.set_ylabel("Price (USD)")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator())

# Add 200-day moving average (a classic technical indicator)
ma_200 = df_clean["Close"].rolling(window=200).mean()
ax1.plot(df_clean.index, ma_200, color="#FF5722", linewidth=1.5,
         linestyle="--", label="200-Day MA", alpha=0.8)
ax1.legend()

## Plot 2: Daily Trading Volume
ax2 = axes[0, 1]
ax2.bar(df_clean.index, df_clean["Volume"] / 1e6,
        color="#4CAF50", alpha=0.6, width=1.0)
ax2.set_title("Daily Trading Volume")
ax2.set_ylabel("Volume (Millions)")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator())

## Plot 3: Daily Returns Distribution
ax3 = axes[1, 0]
daily_returns = df_clean["Close"].pct_change().dropna() * 100  # Convert to percentage
ax3.hist(daily_returns, bins=80, color="#9C27B0", alpha=0.7, edgecolor="white")
ax3.axvline(x=0, color="red", linestyle="--", linewidth=1.5, label="Zero Return")
ax3.axvline(x=daily_returns.mean(), color="orange", linestyle="-",
            linewidth=1.5, label=f"Mean: {daily_returns.mean():.2f}%")
ax3.set_title("Distribution of Daily Returns (%)")
ax3.set_xlabel("Daily Return (%)")
ax3.set_ylabel("Frequency")
ax3.legend()

# Annotate with skewness and kurtosis
skew = daily_returns.skew()
kurt = daily_returns.kurtosis()
ax3.text(0.02, 0.95, f"Skewness: {skew:.3f}\nKurtosis: {kurt:.3f}",
         transform=ax3.transAxes, verticalalignment="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

## Plot 4: Correlation Heatmap
ax4 = axes[1, 1]
correlation_matrix = df_clean.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)
sns.heatmap(
    correlation_matrix,
    ax=ax4,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
ax4.set_title("Feature Correlation Matrix")

## Plot 5: OHLC Candlestick-style (last 60 trading days)
ax5 = axes[2, 0]
recent = df_clean.tail(60).copy()
colors = ["#4CAF50" if close >= open_ else "#F44336"
          for close, open_ in zip(recent["Close"], recent["Open"])]
# Plot High-Low range as thin lines
for i, (date, row) in enumerate(recent.iterrows()):
    ax5.plot([date, date], [row["Low"], row["High"]], color=colors[i],
             linewidth=0.8, alpha=0.7)
# Plot Open-Close range as thicker candle body
for i, (date, row) in enumerate(recent.iterrows()):
    ax5.plot([date, date], [row["Open"], row["Close"]], color=colors[i],
             linewidth=4.0, alpha=0.9)
ax5.set_title("OHLC Price Chart (Last 60 Trading Days)")
ax5.set_ylabel("Price (USD)")
ax5.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

## Plot 6: Rolling 30-Day Volatility
ax6 = axes[2, 1]
rolling_vol = daily_returns.rolling(30).std()
ax6.plot(rolling_vol.index, rolling_vol, color="#FF9800", linewidth=1.2)
ax6.fill_between(rolling_vol.index, rolling_vol, alpha=0.2, color="#FF9800")
ax6.set_title("30-Day Rolling Volatility (Std Dev of Daily Returns %)")
ax6.set_ylabel("Volatility (%)")
ax6.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax6.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
eda_path = CONFIG["OUTPUT_DIR"] / f"{CONFIG['TICKER']}_eda.png"
plt.savefig(eda_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"EDA plot saved to: {eda_path}")

# STATISTICAL INSIGHTS
print("\n" + "="*60)
print("  KEY STATISTICAL INSIGHTS")
print("="*60)
print(f"\n Price Range    : ${df_clean['Close'].min():.2f}  ${df_clean['Close'].max():.2f}")
print(f" Avg Close      : ${df_clean['Close'].mean():.2f}")
print(f" Avg Volume     : {df_clean['Volume'].mean():,.0f} shares/day")
print(f" Max 1-Day Drop : {daily_returns.min():.2f}%")
print(f" Max 1-Day Gain : {daily_returns.max():.2f}%")
print(f" Annualized Vol : {daily_returns.std() * np.sqrt(252):.2f}%")
print(f"\n Correlation(Open, Close)  : {df_clean['Open'].corr(df_clean['Close']):.4f}")
print(f" Correlation(High, Close)  : {df_clean['High'].corr(df_clean['Close']):.4f}")
print(f" Correlation(Volume, Close): {df_clean['Volume'].corr(df_clean['Close']):.4f}")



In [ ]:
# Feature engineering
# Feature engineering is where domain expertise meets data
# science. The features we create here directly determine
# the ceiling of model performance.

def engineer_features(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Transform raw OHLCV data into a rich ML-ready feature matrix.

    Features Created
    ----------------
    Lag Features       : Close price N days ago (memory of recent prices)
    Rolling Statistics : Moving averages, rolling std dev
    Price Ratios       : Intraday range, close-to-open spread
    Momentum Features  : Rate of change, % daily return
    Volume Features    : Volume change, volume vs rolling average
    Target Variable    : Next day's Close (shifted by -1)

    Parameters
    ----------
    df : pd.DataFrame
        Clean OHLCV DataFrame
    config : dict
        Project configuration dictionary

    Returns
    -------
    pd.DataFrame
        Feature-rich DataFrame with target column
    """
    logger.info("Starting feature engineering...")

    # Work on a copy so the raw data stays unchanged.
    feat = df.copy()

    ## Group 1: LAG FEATURES
    # These give the model "memory"  what happened in recent days-
    # lag=1 means: "what was yesterday's Close-"
    for lag in config["LAG_DAYS"]:
        feat[f"Close_lag{lag}"] = feat["Close"].shift(lag)
        feat[f"Volume_lag{lag}"] = feat["Volume"].shift(lag)

    logger.info(f"  Created {len(config['LAG_DAYS']) * 2} lag features.")

    ## Group 2: ROLLING WINDOW STATISTICS
    # Rolling mean smooths out day-to-day noise and captures the trend direction
    # Rolling std captures recent volatility (how much prices are fluctuating)
    for window in config["ROLLING_WINDOWS"]:
        # Rolling mean (simple moving average)
        feat[f"Close_ma{window}"] = feat["Close"].rolling(window=window).mean()

        # Rolling standard deviation (volatility measure)
        feat[f"Close_std{window}"] = feat["Close"].rolling(window=window).std()

        # Price position relative to moving average
        # > 1.0 means price is above the moving average (bullish signal)
        # < 1.0 means price is below the moving average (bearish signal)
        feat[f"Price_to_ma{window}"] = feat["Close"] / feat[f"Close_ma{window}"]

    logger.info(f"  Created {len(config['ROLLING_WINDOWS']) * 3} rolling statistics features.")

    ## Group 3: INTRADAY PRICE STRUCTURE FEATURES
    # Intraday price structure.

    # High-Low range: How wide was today's price swing-
    # Wide range = high uncertainty/volatility that day
    feat["HL_range"] = feat["High"] - feat["Low"]

    # Close relative to daily range (0 = closed at day's low, 1 = at day's high)
    # Formula: (Close - Low) / (High - Low)
    # This is called the "Stochastic %K" in technical analysis
    feat["Close_pct_of_range"] = (
        (feat["Close"] - feat["Low"]) / (feat["High"] - feat["Low"] + 1e-8)
    )

    # Open-to-Close spread: Did the price go up or down during the day-
    feat["OC_spread"] = feat["Close"] - feat["Open"]

    # Open gap: How much did price "gap" from yesterday's close to today's open-
    # Large gaps indicate news events overnight
    feat["Open_gap"] = feat["Open"] - feat["Close"].shift(1)

    ## Group 4: MOMENTUM AND RETURN FEATURES
    # Momentum measures the rate of price change

    # Daily percentage return: (today - yesterday) / yesterday
    # pct_change() does this automatically
    feat["Daily_return"] = feat["Close"].pct_change()

    # Rate of change over 5 and 10 days (medium-term momentum)
    feat["ROC_5"] = feat["Close"].pct_change(periods=5)
    feat["ROC_10"] = feat["Close"].pct_change(periods=10)

    ## Group 5: VOLUME FEATURES
    # Volume context: Was today's volume higher or lower than usual-
    # High volume on a price move confirms the move; low volume = unreliable

    # Normalize volume by its 20-day rolling average
    # > 1.0 means above-average volume (high activity)
    # < 1.0 means below-average volume (quiet day)
    feat["Volume_ratio_20d"] = feat["Volume"] / feat["Volume"].rolling(20).mean()

    # Volume change from yesterday
    feat["Volume_change"] = feat["Volume"].pct_change()

    # TARGET VARIABLE
    # This is the KEY step: shift Close by -1 to create "next day's Close"
    # shift(-1) moves all values up by 1 row
    # Row for 2024-01-02 now contains 2024-01-03's Close as the target
    feat["Target_Close"] = feat["Close"].shift(-1)

    logger.info("  Created target variable: Next day's closing price.")

    # CLEAN UP
    # Remove rows with NaN values (created by rolling windows and lag features)
    # The first N rows will have NaN for rolling/lag features  drop them
    rows_before = len(feat)
    feat.dropna(inplace=True)
    rows_dropped = rows_before - len(feat)

    logger.info(f"  Dropped {rows_dropped} rows with NaN (from window/lag warmup).")
    logger.info(f"  Final feature matrix: {feat.shape[0]:,} rows  {feat.shape[1]} columns")

    return feat


# Execute feature engineering
df_features = engineer_features(df_clean, CONFIG)

# DISPLAY FEATURE SUMMARY
feature_cols = [col for col in df_features.columns if col != "Target_Close"]

print("\n" + "="*60)
print("  FEATURE ENGINEERING SUMMARY")
print("="*60)
print(f"\n Feature matrix shape : {df_features[feature_cols].shape}")
print(f" Target column        : 'Target_Close' (next day's Close)")
print(f"\n All features ({len(feature_cols)} total):")
for i, col in enumerate(feature_cols, 1):
    print(f"   {i:2d}. {col}")

print("\n Sample of engineered features (last 3 rows):")
display(df_features[feature_cols + ["Target_Close"]].tail(3).round(4))



In [ ]:
# Time-aware train/test split
# "If your test set contains data from before your training
# set, your evaluation is meaningless."  Core ML principle

def prepare_train_test_split(
    df: pd.DataFrame,
    train_ratio: float,
    feature_cols: list,
    target_col: str = "Target_Close",
) -> tuple:
    """
    Perform a chronological (time-aware) train/test split with feature scaling.

    Why chronological?
    ------------------
    In time series, the future cannot influence the past. If we randomly
    split, the model trains on tomorrow and predicts yesterday  an
    impossible and useless setup. Chronological split ensures the model
    only trains on data that came before the test period.

    Parameters
    ----------
    df : pd.DataFrame
        Feature-engineered DataFrame
    train_ratio : float
        Proportion of data for training (e.g., 0.80 = 80%)
    feature_cols : list
        List of feature column names (X)
    target_col : str
        Name of the target column (y)

    Returns
    -------
    tuple : (X_train, X_test, y_train, y_test, scaler, split_date)
    """
    logger.info("Preparing chronological train/test split...")

    ## Step 1: CHRONOLOGICAL SPLIT
    # Calculate the split index (e.g., 80% of rows = training)
    split_idx = int(len(df) * train_ratio)

    # Everything before split_idx is training data
    # Everything from split_idx onward is test data
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    split_date = test_df.index[0]

    logger.info(f"  Training period: {train_df.index[0].date()}  {train_df.index[-1].date()} ({len(train_df):,} days)")
    logger.info(f"  Testing period : {test_df.index[0].date()}  {test_df.index[-1].date()} ({len(test_df):,} days)")

    ## Step 2: SEPARATE FEATURES (X) AND TARGET (y)
    X_train = train_df[feature_cols].values   # numpy array for sklearn
    y_train = train_df[target_col].values
    X_test = test_df[feature_cols].values
    y_test = test_df[target_col].values

    ## Step 3: FEATURE SCALING
    # StandardScaler: transforms each feature to zero mean and unit variance
    # Formula: z = (x - mean) / std_dev
    #
    # Important RULE: Fit only on training data, then transform both sets.
    # Fitting on test data would leak future information.
    scaler = StandardScaler()

    # fit_transform() on training: learns the mean/std AND applies transformation
    X_train_scaled = scaler.fit_transform(X_train)

    # transform() on test: uses the same mean/std learned from training data
    # (does not relearn statistics from the test period)
    X_test_scaled = scaler.transform(X_test)

    logger.info("  Feature scaling applied (StandardScaler fitted on training data only).")
    logger.info(f"  X_train shape: {X_train_scaled.shape}")
    logger.info(f"  X_test shape : {X_test_scaled.shape}")

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, split_date, train_df, test_df


# Execute preprocessing
feature_cols = [col for col in df_features.columns if col != "Target_Close"]

(X_train, X_test, y_train, y_test,
 fitted_scaler, split_date,
 train_df, test_df) = prepare_train_test_split(
    df=df_features,
    train_ratio=CONFIG["TRAIN_RATIO"],
    feature_cols=feature_cols,
    target_col="Target_Close",
)

# VISUALIZE THE SPLIT
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(train_df.index, train_df["Close"],
        color="#2196F3", linewidth=1.2, label=f"Training Data ({len(train_df):,} days)", alpha=0.9)
ax.plot(test_df.index, test_df["Close"],
        color="#FF5722", linewidth=1.5, label=f"Test Data ({len(test_df):,} days)", alpha=0.9)

ax.axvline(x=split_date, color="gold", linewidth=2.5,
           linestyle="--", label=f"Train/Test Split: {split_date.date()}")

ax.fill_between(train_df.index, train_df["Close"], alpha=0.08, color="#2196F3")
ax.fill_between(test_df.index, test_df["Close"], alpha=0.08, color="#FF5722")

ax.set_title(f"{CONFIG['TICKER']}  Chronological Train/Test Split", fontsize=16)
ax.set_ylabel("Closing Price (USD)")
ax.legend(loc="upper left", fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
split_path = CONFIG["OUTPUT_DIR"] / f"{CONFIG['TICKER']}_train_test_split.png"
plt.savefig(split_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"\n Cell 5 complete  Data split at {split_date.date()}")
print(f"    Training samples : {X_train.shape[0]:,}")
print(f"    Testing samples  : {X_test.shape[0]:,}")


In [ ]:
# Linear regression model
# Linear Regression is always the baseline model. If a more
# complex model can't beat the linear baseline, it's either
# overfitting or the features aren't informative enough.

def train_linear_regression(X_train, y_train) -> LinearRegression:
    """
    Train a Linear Regression model.

    Parameters
    ----------
    X_train : np.ndarray
        Scaled training features
    y_train : np.ndarray
        Training target values (next day's Close)

    Returns
    -------
    LinearRegression
        Trained model object
    """
    logger.info("Training Linear Regression model...")

    model = LinearRegression(
        fit_intercept=True,   # Include the intercept term
        copy_X=True,          # Don't overwrite input data
        n_jobs=-1,            # Use all CPU cores for computation
    )

    # .fit() is where the math happens: solves the OLS equation
    # It computes the coefficients that minimize squared error
    model.fit(X_train, y_train)

    logger.info("  Linear Regression training complete.")
    logger.info(f"  Intercept: {model.intercept_:.4f}")
    logger.info(f"  Number of coefficients: {len(model.coef_)}")

    return model


## Train model
lr_model = train_linear_regression(X_train, y_train)

## Generate predictions
# Use the trained model to predict on unseen test data
lr_train_preds = lr_model.predict(X_train)   # For diagnosing overfitting
lr_test_preds = lr_model.predict(X_test)    # The actual evaluation predictions

## Inspect coefficients
# Understanding which features the model relies on most
coeff_df = pd.DataFrame({
    "Feature"    : feature_cols,
    "Coefficient": lr_model.coef_,
    "Abs_Impact" : np.abs(lr_model.coef_),
}).sort_values("Abs_Impact", ascending=False)

print("\n" + "="*60)
print("  LINEAR REGRESSION  TOP 10 MOST INFLUENTIAL FEATURES")
print("="*60)
print(f"\n  Intercept: {lr_model.intercept_:.4f}\n")
display(coeff_df.head(10).round(4))

print("\n Interpretation:")
print("   Positive coefficient  feature pushes prediction UP")
print("   Negative coefficient  feature pushes prediction DOWN")
print("   Larger absolute value  stronger influence on prediction")


In [ ]:
# CELL 7: RANDOM FOREST REGRESSOR MODEL
# Random Forest handles non-linear relationships that Linear
# Regression completely misses. It also naturally provides
# feature importance scores  a huge advantage for analysis.

def train_random_forest(X_train, y_train, config: dict) -> RandomForestRegressor:
    """
    Train a Random Forest Regressor with small hyperparameters.

    Key Hyperparameters Explained
    -----------------------------
    n_estimators  : More trees = better performance but slower training.
                    200 is a good balance for daily stock data.
    max_depth     : Limits how deep each tree can grow. Shallow trees
                    (smaller max_depth) generalize better and don't overfit.
    min_samples_leaf: A node becomes a leaf only if it has this many samples.
                    Higher values = smoother predictions = less overfitting.
    max_features  : Each split only considers a random fraction of features.
                    "sqrt" means sqrt(n_features) features per split.

    Parameters
    ----------
    X_train : np.ndarray
        Training features (scaling not required for RF, but we use scaled
        data for consistency with Linear Regression comparison)
    y_train : np.ndarray
        Training target values
    config : dict
        Project configuration

    Returns
    -------
    RandomForestRegressor
        Trained model object
    """
    logger.info("Training Random Forest Regressor...")
    logger.info(f"  n_estimators: {config['RF_N_ESTIMATORS']}, max_depth: {config['RF_MAX_DEPTH']}")

    model = RandomForestRegressor(
        n_estimators = config["RF_N_ESTIMATORS"],
        max_depth = config["RF_MAX_DEPTH"],
        min_samples_leaf = config["RF_MIN_SAMPLES_LEAF"],
        max_features = "sqrt",       # Classic Breiman recommendation for RF
        bootstrap = True,         # Use bootstrap sampling (bagging)
        oob_score = True,         # Out-of-Bag score: free internal validation
        n_jobs = -1,           # Parallelize across all CPU cores
        random_state = config["RF_RANDOM_STATE"],
        verbose = 0,
    )

    model.fit(X_train, y_train)

    logger.info("  Random Forest training complete.")
    logger.info(f"  Out-of-Bag R2 Score: {model.oob_score_:.4f}")
    logger.info(f"  (OOB score is a free estimate of generalization performance)")

    return model


## Train model
rf_model = train_random_forest(X_train, y_train, CONFIG)

## Generate predictions
rf_train_preds = rf_model.predict(X_train)
rf_test_preds = rf_model.predict(X_test)

print("\n" + "="*60)
print("  RANDOM FOREST  MODEL SUMMARY")
print("="*60)
print(f"\n  Trees trained       : {CONFIG['RF_N_ESTIMATORS']}")
print(f"  Max tree depth      : {CONFIG['RF_MAX_DEPTH']}")
print(f"  Min samples/leaf    : {CONFIG['RF_MIN_SAMPLES_LEAF']}")
print(f"  OOB R2 Score        : {rf_model.oob_score_:.4f}")
print(f"\n   OOB score: {rf_model.oob_score_:.1%} of test variance explained")
print(f"     (estimated from training data  no test set needed)")


In [ ]:
# Model evaluation
# Evaluation must be honest. Always measure on held-out test
# data that the model has never seen during training.

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str) -> dict:
    """
    Compute a full suite of regression evaluation metrics.

    Parameters
    ----------
    y_true : np.ndarray
        Actual closing prices from test set
    y_pred : np.ndarray
        Model's predicted closing prices
    model_name : str
        Name label for logging/display

    Returns
    -------
    dict
        Dictionary of metric name to value
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    # Mean Absolute Percentage Error (MAPE)
    # How large is the error as a percentage of the actual price?
    # Useful for comparing across stocks with different price levels
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

    # Directional accuracy: did the model correctly predict UP vs DOWN?
    # Even if the exact price is wrong, getting the direction right matters
    actual_direction = np.sign(np.diff(y_true))    # +1 if price went up, -1 if down
    pred_direction = np.sign(np.diff(y_pred))
    directional_acc = np.mean(actual_direction == pred_direction) * 100

    metrics = {
        "MAE":               mae,
        "RMSE":              rmse,
        "R2":                r2,
        "MAPE (%)":          mape,
        "Directional Acc (%)": directional_acc,
    }

    logger.info(f"\n{model_name} Test Metrics:")
    for name, val in metrics.items():
        logger.info(f"  {name:<25} {val:.4f}")

    return metrics


# EVALUATE BOTH MODELS

# Evaluate on test set (never report training set performance as your main result)
lr_metrics = compute_metrics(y_test, lr_test_preds, "Linear Regression")
rf_metrics = compute_metrics(y_test, rf_test_preds, "Random Forest")

# Also compute train metrics to detect overfitting
lr_train_metrics = compute_metrics(y_train, lr_train_preds, "Linear Regression (Train)")
rf_train_metrics = compute_metrics(y_train, rf_train_preds, "Random Forest (Train)")

# BUILD COMPARISON TABLE
results_df = pd.DataFrame({
    "Metric"         : list(lr_metrics.keys()),
    "Linear Reg (Train)": list(lr_train_metrics.values()),
    "Linear Reg (Test)": list(lr_metrics.values()),
    "Random Forest (Train)": list(rf_train_metrics.values()),
    "Random Forest (Test)": list(rf_metrics.values()),
})

print("\n" + "="*70)
print("  MODEL EVALUATION RESULTS  TRAIN VS TEST")
print("="*70)
display(results_df.round(4))

print("\n How to Read This Table:")
print("   If Train score >> Test score  Model is OVERFITTING (memorizing training data)")
print("   If Train score  Test score   Model is GENERALIZING properly")
print("   Lower MAE/RMSE = better | Higher R = better | Higher Directional Acc = better")


In [ ]:
# CELL 9: VISUALIZATION  ACTUAL VS PREDICTED PRICES
# A great visualization communicates the model's performance
# better than any table of numbers.

# Rebuild test dates for the x-axis
test_dates = test_df.index[:len(y_test)]

fig, axes = plt.subplots(2, 1, figsize=(18, 12))
fig.suptitle(
    f"{CONFIG['TICKER']}  Actual vs Predicted Closing Prices",
    fontsize=18, fontweight="bold"
)

## Plot 1: LINEAR REGRESSION PREDICTIONS
ax1 = axes[0]
ax1.plot(test_dates, y_test, color="#2196F3", linewidth=1.5,
         label="Actual Close", zorder=3, alpha=0.9)
ax1.plot(test_dates, lr_test_preds, color="#FF5722", linewidth=1.3,
         linestyle="--", label=f"LR Predicted  (MAE=${lr_metrics['MAE']:.2f}, R2={lr_metrics['R2']:.4f})",
         zorder=2, alpha=0.85)

# Shade the error region
ax1.fill_between(test_dates, y_test, lr_test_preds,
                 alpha=0.15, color="#FF5722", label="Prediction Error")
ax1.set_title("Linear Regression", fontsize=14)
ax1.set_ylabel("Price (USD)")
ax1.legend(loc="upper left")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.AutoDateLocator())

## Plot 2: RANDOM FOREST PREDICTIONS
ax2 = axes[1]
ax2.plot(test_dates, y_test, color="#2196F3", linewidth=1.5,
         label="Actual Close", zorder=3, alpha=0.9)
ax2.plot(test_dates, rf_test_preds, color="#4CAF50", linewidth=1.3,
         linestyle="--", label=f"RF Predicted  (MAE=${rf_metrics['MAE']:.2f}, R2={rf_metrics['R2']:.4f})",
         zorder=2, alpha=0.85)

ax2.fill_between(test_dates, y_test, rf_test_preds,
                 alpha=0.12, color="#4CAF50", label="Prediction Error")
ax2.set_title("Random Forest Regressor", fontsize=14)
ax2.set_ylabel("Price (USD)")
ax2.legend(loc="upper left")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.xaxis.set_major_locator(mdates.AutoDateLocator())

plt.tight_layout()
pred_path = CONFIG["OUTPUT_DIR"] / f"{CONFIG['TICKER']}_predictions.png"
plt.savefig(pred_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"Prediction plot saved to: {pred_path}")


In [ ]:
# Model comparison dashboard
# Compare errors, residuals, and headline metrics side by side.

fig = plt.figure(figsize=(20, 14))
fig.suptitle(f"{CONFIG['TICKER']}  Model Performance Dashboard", fontsize=20, fontweight="bold")

# Create a 3x3 grid of subplots
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)

## Panel 1: LR Scatter  Actual vs Predicted
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_test, lr_test_preds, alpha=0.4, s=10, color="#FF5722")
min_val, max_val = min(y_test.min(), lr_test_preds.min()), max(y_test.max(), lr_test_preds.max())
ax1.plot([min_val, max_val], [min_val, max_val], "b--", lw=1.5, label="Perfect prediction")
ax1.set_xlabel("Actual Price")
ax1.set_ylabel("Predicted Price")
ax1.set_title(f"LR: Actual vs Predicted\nR={lr_metrics['R2']:.4f}")
ax1.legend(fontsize=9)

## Panel 2: RF Scatter  Actual vs Predicted
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_test, rf_test_preds, alpha=0.4, s=10, color="#4CAF50")
ax2.plot([min_val, max_val], [min_val, max_val], "b--", lw=1.5, label="Perfect prediction")
ax2.set_xlabel("Actual Price")
ax2.set_ylabel("Predicted Price")
ax2.set_title(f"RF: Actual vs Predicted\nR={rf_metrics['R2']:.4f}")
ax2.legend(fontsize=9)

## Panel 3: Metrics Bar Chart
ax3 = fig.add_subplot(gs[0, 2])
metric_names = ["MAE", "RMSE", "MAPE (%)"]
lr_vals = [lr_metrics[m] for m in metric_names]
rf_vals = [rf_metrics[m] for m in metric_names]
x = np.arange(len(metric_names))
bars1 = ax3.bar(x - 0.2, lr_vals, 0.35, label="Linear Reg", color="#FF5722", alpha=0.8)
bars2 = ax3.bar(x + 0.2, rf_vals, 0.35, label="Random Forest", color="#4CAF50", alpha=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels(metric_names, fontsize=9)
ax3.set_title("Error Metrics Comparison\n(Lower = Better)")
ax3.legend(fontsize=9)
for bar in bars1: ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
for bar in bars2: ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

## Panel 4: LR Residuals Distribution
ax4 = fig.add_subplot(gs[1, 0])
lr_residuals = y_test - lr_test_preds
ax4.hist(lr_residuals, bins=50, color="#FF5722", alpha=0.7, edgecolor="white")
ax4.axvline(x=0, color="blue", linestyle="--", lw=1.5)
ax4.axvline(x=lr_residuals.mean(), color="orange", linestyle="-", lw=1.5,
            label=f"Mean: {lr_residuals.mean():.2f}")
ax4.set_title("LR: Residuals Distribution\n(Should be centered at 0)")
ax4.set_xlabel("Prediction Error ($)")
ax4.legend(fontsize=9)

## Panel 5: RF Residuals Distribution
ax5 = fig.add_subplot(gs[1, 1])
rf_residuals = y_test - rf_test_preds
ax5.hist(rf_residuals, bins=50, color="#4CAF50", alpha=0.7, edgecolor="white")
ax5.axvline(x=0, color="blue", linestyle="--", lw=1.5)
ax5.axvline(x=rf_residuals.mean(), color="orange", linestyle="-", lw=1.5,
            label=f"Mean: {rf_residuals.mean():.2f}")
ax5.set_title("RF: Residuals Distribution\n(Should be centered at 0)")
ax5.set_xlabel("Prediction Error ($)")
ax5.legend(fontsize=9)

## Panel 6: R Comparison Bar
ax6 = fig.add_subplot(gs[1, 2])
r2_values = [lr_metrics["R2"], rf_metrics["R2"]]
colors = ["#FF5722", "#4CAF50"]
bars = ax6.bar(["Linear\nRegression", "Random\nForest"], r2_values, color=colors, alpha=0.8, width=0.5)
ax6.set_ylim(0, 1.05)
ax6.set_title("R2 Score Comparison\n(Higher = Better, Max = 1.0)")
ax6.axhline(y=1.0, color="gold", linestyle="--", alpha=0.5, label="Perfect = 1.0")
for bar, val in zip(bars, r2_values):
    ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax6.legend(fontsize=9)

## Panel 7: LR Residuals Over Time
ax7 = fig.add_subplot(gs[2, :2])
ax7.plot(test_dates, lr_residuals, color="#FF5722", alpha=0.6, linewidth=0.8, label="LR Residuals")
ax7.plot(test_dates, rf_residuals, color="#4CAF50", alpha=0.6, linewidth=0.8, label="RF Residuals")
ax7.axhline(y=0, color="white", linestyle="--", linewidth=1.0)
ax7.fill_between(test_dates, lr_residuals, alpha=0.1, color="#FF5722")
ax7.fill_between(test_dates, rf_residuals, alpha=0.1, color="#4CAF50")
ax7.set_title("Residuals Over Time (Prediction Error by Date)")
ax7.set_ylabel("Error ($)")
ax7.legend()
ax7.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

## Panel 8: Directional Accuracy
ax8 = fig.add_subplot(gs[2, 2])
dir_values = [lr_metrics["Directional Acc (%)"], rf_metrics["Directional Acc (%)"], 50]
dir_labels = ["Linear\nRegression", "Random\nForest", "Random\nGuess\n(Baseline)"]
dir_colors = ["#FF5722", "#4CAF50", "#9E9E9E"]
bars = ax8.bar(dir_labels, dir_values, color=dir_colors, alpha=0.8, width=0.5)
ax8.set_ylim(0, 100)
ax8.set_title("Directional Accuracy (%)\n(Did we predict UP/DOWN correctly-)")
ax8.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="50% = random guess")
for bar, val in zip(bars, dir_values):
    ax8.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax8.legend(fontsize=9)

dashboard_path = CONFIG["OUTPUT_DIR"] / f"{CONFIG['TICKER']}_dashboard.png"
plt.savefig(dashboard_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"Dashboard saved to: {dashboard_path}")


In [ ]:
# Feature importance
# Feature importance reveals which features the model relies
# on most. This is one of the most valuable insights for
# domain experts and business stakeholders.

def plot_feature_importance(model, feature_names: list, model_name: str, top_n: int = 20):
    """
    Plot horizontal bar chart of feature importances from Random Forest.

    Random Forest computes importance as the total reduction in impurity
    (variance, for regression) achieved by splitting on each feature
    across all trees. Features with high importance drove more splits
    and thus had more predictive power.

    Parameters
    ----------
    model : RandomForestRegressor
        Trained Random Forest model
    feature_names : list
        List of feature column names
    model_name : str
        Display name for the plot title
    top_n : int
        Show top N most important features
    """
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature"   : feature_names,
        "Importance": importances,
    }).sort_values("Importance", ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(12, 8))

    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importance_df)))
    bars = ax.barh(
        importance_df["Feature"],
        importance_df["Importance"],
        color=colors,
        edgecolor="white",
        linewidth=0.5,
    )

    # Add value labels to bars
    for bar, val in zip(bars, importance_df["Importance"]):
        ax.text(
            bar.get_width() + 0.0005,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}",
            va="center",
            fontsize=9,
        )

    ax.set_xlabel("Feature Importance Score", fontsize=13)
    ax.set_title(
        f"{model_name}  Top {top_n} Feature Importances\n"
        f"({CONFIG['TICKER']} Stock Prediction)",
        fontsize=15, fontweight="bold"
    )
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    fi_path = CONFIG["OUTPUT_DIR"] / f"{CONFIG['TICKER']}_feature_importance.png"
    plt.savefig(fi_path, dpi=150, bbox_inches="tight")
    plt.show()
    logger.info(f"Feature importance plot saved to: {fi_path}")

    return importance_df.sort_values("Importance", ascending=False)


importance_results = plot_feature_importance(
    model=rf_model,
    feature_names=feature_cols,
    model_name="Random Forest Regressor",
    top_n=20,
)

print("\n Top 10 Most Important Features:")
display(importance_results.head(10).round(5))

print("\n Interpretation Guide:")
print("   Close_lag1        Yesterday's price is usually most predictive")
print("   Close_ma5         5-day moving average captures short-term trend")
print("   Close_ma20        20-day MA captures medium-term trend direction")
print("   HL_range          Intraday volatility can predict next day move")
print("   Volume_ratio_20d  Volume vs average signals conviction behind moves")


In [ ]:
# Next-day prediction
# This is the real-world application: use the most recent
# available data to forecast what tomorrow's price will be.
# This is exactly what a production inference pipeline does.

def predict_next_day(
    df_features: pd.DataFrame,
    feature_cols: list,
    scaler: StandardScaler,
    lr_model: LinearRegression,
    rf_model: RandomForestRegressor,
    ticker: str,
) -> pd.DataFrame:
    """
    Use the very last row of data to predict tomorrow's closing price.

    In production, this function would be called every evening after
    market close with the day's freshly computed features.

    Parameters
    ----------
    df_features : pd.DataFrame
        Full feature-engineered DataFrame
    feature_cols : list
        Feature column names
    scaler : StandardScaler
        The fitted scaler fitted on training data (use the fitted scaler)
    lr_model : LinearRegression
        Trained Linear Regression model
    rf_model : RandomForestRegressor
        Trained Random Forest model
    ticker : str
        Stock ticker for display

    Returns
    -------
    pd.DataFrame
        Prediction summary table
    """
    logger.info(f"Generating next-day prediction for {ticker}...")

    # Get the most recent day's data (last row of feature matrix)
    last_row = df_features[feature_cols].iloc[[-1]]   # Keep as DataFrame (not Series)
    last_date = df_features.index[-1]

    # Calculate next business day (skip weekends)
    next_day = last_date + pd.offsets.BDay(1)

    # Scale using the same scaler fitted on training data
    # do not refit the scaler on new data  that changes the reference baseline
    last_row_scaled = scaler.transform(last_row)

    # Generate model predictions.
    lr_prediction = lr_model.predict(last_row_scaled)[0]
    rf_prediction = rf_model.predict(last_row_scaled)[0]

    # Simple average of both model outputs.
    ensemble_prediction = (lr_prediction + rf_prediction) / 2

    # Current actual close (most recent known)
    current_close = df_features["Close"].iloc[-1]

    # Calculate predicted change
    lr_change = lr_prediction - current_close
    rf_change = rf_prediction - current_close
    ens_change = ensemble_prediction - current_close

    results = pd.DataFrame({
        "Model"              : ["Linear Regression", "Random Forest", "Ensemble (Avg)"],
        "Current Close ($)"  : [f"{current_close:.2f}"] * 3,
        "Predicted Next Close ($)": [f"{lr_prediction:.2f}", f"{rf_prediction:.2f}", f"{ensemble_prediction:.2f}"],
        "Predicted Change ($)": [f"{lr_change:+.2f}", f"{rf_change:+.2f}", f"{ens_change:+.2f}"],
        "Predicted Direction": [
            " UP" if lr_change > 0 else " DOWN",
            " UP" if rf_change > 0 else " DOWN",
            " UP" if ens_change > 0 else " DOWN",
        ]
    })

    print("\n" + "="*65)
    print(f"   PREDICTION FOR: {ticker} on {next_day.date()}")
    print(f"     (Based on data through {last_date.date()})")
    print("="*65)
    display(results)

    # Visual price gauge
    print(f"\n   Price Gauge:")
    print(f"     Today's Close  : ${current_close:.2f}")
    print(f"     LR Prediction  : ${lr_prediction:.2f}  ({lr_change:+.2f})")
    print(f"     RF Prediction  : ${rf_prediction:.2f}  ({rf_change:+.2f})")
    print(f"     Ensemble       : ${ensemble_prediction:.2f}  ({ens_change:+.2f})")

    print(f"\n    DISCLAIMER: This is a model output for educational purposes only.")
    print(f"     Stock markets are inherently unpredictable. Do NOT use this")
    print(f"     for real investment decisions.")

    return results


prediction_results = predict_next_day(
    df_features=df_features,
    feature_cols=feature_cols,
    scaler=fitted_scaler,
    lr_model=lr_model,
    rf_model=rf_model,
    ticker=CONFIG["TICKER"],
)

# FINAL SUMMARY
print("\n" + "="*65)
print("   FINAL PROJECT SUMMARY")
print("="*65)
print(f"\n  Stock Analyzed     : {CONFIG['TICKER']}")
print(f"  Total Trading Days : {len(df_features):,}")
print(f"  Features Engineered: {len(feature_cols)}")
print(f"\n  {'Model':<22} {'MAE':>8} {'RMSE':>8} {'R':>8} {'Dir Acc':>10}")
print(f"  {'-'*58}")
print(f"  {'Linear Regression':<22} ${lr_metrics['MAE']:>6.2f}  ${lr_metrics['RMSE']:>6.2f}  "
      f"{lr_metrics['R2']:>7.4f}  {lr_metrics['Directional Acc (%)']:>8.1f}%")
print(f"  {'Random Forest':<22} ${rf_metrics['MAE']:>6.2f}  ${rf_metrics['RMSE']:>6.2f}  "
      f"{rf_metrics['R2']:>7.4f}  {rf_metrics['Directional Acc (%)']:>8.1f}%")

winner = "Random Forest" if rf_metrics["R2"] > lr_metrics["R2"] else "Linear Regression"
print(f"\n   Best Model by R  : {winner}")
print(f"\n   Outputs saved to  : {CONFIG['OUTPUT_DIR'].resolve()}/")
print(f"     - {CONFIG['TICKER']}_eda.png")
print(f"     - {CONFIG['TICKER']}_train_test_split.png")
print(f"     - {CONFIG['TICKER']}_predictions.png")
print(f"     - {CONFIG['TICKER']}_dashboard.png")
print(f"     - {CONFIG['TICKER']}_feature_importance.png")
